# Recipe Gemma: results analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/results_analysis.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/results_analysis.ipynb)

Loads whatever is in `results/`: `train_smoke.json`, `train_smoke_log.csv`, `train_full*.json/csv`,
`eval_metrics.json`, `judge_preference.json` and `human_preference.json`. It plots loss curves and
metric bars and prints the tables. Missing files are skipped. The committed copy has no saved
outputs, so run it after training/eval to see the plots.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# Works from the repo root or from notebooks/.
RESULTS = next((p for p in [Path("results"), Path("../results")] if p.is_dir()), Path("results"))
print("results dir:", RESULTS.resolve(), "| exists:", RESULTS.is_dir())

# Categorical colors in fixed order (one per run/model, never cycled).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e4e3df", "grid.linewidth": 0.8, "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=SERIES),
})


def load_json(name):
    p = RESULTS / name
    if not p.exists():
        print(f"(missing) {p}")
        return None
    return json.loads(p.read_text(encoding="utf-8"))


def load_csv(name):
    p = RESULTS / name
    if not p.exists():
        print(f"(missing) {p}")
        return None
    return pd.read_csv(p)


def flatten(d, prefix=""):
    """Nested dict -> {"a.b.c": scalar}."""
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}{k}."))
    elif isinstance(d, list):
        if all(isinstance(x, (int, float, str, bool)) or x is None for x in d) and len(d) <= 10:
            out[prefix[:-1]] = d
    else:
        out[prefix[:-1]] = d
    return out


def numeric_table(d, drop=("meta", "config")):
    """Flatten a results dict into a (key, value) table of numeric leaves."""
    if d is None:
        return pd.DataFrame(columns=["key", "value"])
    d = {k: v for k, v in d.items() if k not in drop} if isinstance(d, dict) else d
    rows = [(k, v) for k, v in flatten(d).items()
            if isinstance(v, (int, float)) and not isinstance(v, bool)]
    return pd.DataFrame(rows, columns=["key", "value"])

## Training runs

In [ ]:
runs = {}
for p in sorted(RESULTS.glob("train_*.json")):
    runs[p.stem] = json.loads(p.read_text(encoding="utf-8"))
if not runs:
    print("no results/train_*.json yet")

summary = []
for name, r in runs.items():
    t, m, ppl = r.get("training", {}), r.get("model", {}), r.get("val_perplexity", {})
    summary.append({
        "run": name,
        "preset": m.get("preset"),
        "lora_rank": m.get("lora_rank"),
        "trainable_%": round(m.get("trainable_pct", float("nan")), 3),
        "steps": t.get("steps_run"),
        "best_step": t.get("best_step"),
        "best_val_loss": t.get("best_val_loss"),
        "val_ppl_base": (ppl.get("base") or {}).get("ppl"),
        "val_ppl_tuned": (ppl.get("fine_tuned") or {}).get("ppl"),
        "sec_per_step": t.get("sec_per_step"),
        "early_stopped": t.get("early_stopped"),
    })
pd.DataFrame(summary)

In [ ]:
# Loss curves: CSVLogger rows are one per validation interval (eval_every steps).
logs = {}
for p in sorted(RESULTS.glob("train_*_log.csv")):
    df = pd.read_csv(p)
    run = p.stem.removesuffix("_log")
    every = runs.get(run, {}).get("training", {}).get("eval_every", 1)
    df["step"] = (df["epoch"] + 1) * every
    logs[run] = df
    print(f"{p.name}: {len(df)} rows, columns={list(df.columns)}")

if logs:
    fig, axes = plt.subplots(1, len(logs), figsize=(6 * len(logs), 4), squeeze=False)
    for ax, (run, df) in zip(axes[0], logs.items()):
        ax.plot(df["step"], df["loss"], marker="o", markersize=4, label="train loss", color=SERIES[0])
        if "val_loss" in df:
            ax.plot(df["step"], df["val_loss"], marker="o", markersize=4, label="val loss", color=SERIES[1])
        best = runs.get(run, {}).get("training", {}).get("best_step")
        if best:
            ax.axvline(best, color="#8a8984", linestyle="--", linewidth=1)
            ax.annotate("best", (best, ax.get_ylim()[1]), ha="right", va="top", fontsize=9, color="#5f5e5a")
        ax.set_title(f"{run}: loss (response tokens only)")
        ax.set_xlabel("optimizer step")
        ax.set_ylabel("cross-entropy")
        ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print("no results/train_*_log.csv yet")

In [ ]:
# Validation perplexity before vs after LoRA, per run (one axis, same unit).
if summary:
    df = pd.DataFrame(summary).set_index("run")[["val_ppl_base", "val_ppl_tuned"]].dropna(how="all")
    if not df.empty:
        ax = df.rename(columns={"val_ppl_base": "base", "val_ppl_tuned": "LoRA fine-tuned"}).plot.bar(
            figsize=(6, 3.5), color=SERIES[:2], width=0.7, edgecolor="white", linewidth=2, rot=0)
        ax.set_ylabel("validation perplexity (lower is better)")
        ax.set_title("Validation perplexity")
        for c in ax.containers:
            ax.bar_label(c, fmt="%.2f", fontsize=8, color="#5f5e5a")
        ax.legend(frameon=False)
        plt.tight_layout()
        plt.show()

## Evaluation metrics (`eval_metrics.json`)

In [ ]:
eval_metrics = load_json("eval_metrics.json")
eval_tbl = numeric_table(eval_metrics)
if not eval_tbl.empty:
    print(eval_tbl.to_string(index=False))
eval_tbl

In [ ]:
# Metric bars. Assumes keys look like <model>.<metric...> or <task>.<model>.<metric>; the
# grouping falls back to one bar per flattened key if the structure differs.
def metric_bars(tbl, title):
    if tbl.empty:
        print(f"{title}: nothing to plot")
        return
    parts = tbl["key"].str.split(".", n=1, expand=True)
    if parts.shape[1] == 2 and parts[0].nunique() <= len(SERIES):
        piv = tbl.assign(group=parts[0], metric=parts[1]).pivot_table(
            index="metric", columns="group", values="value", aggfunc="first")
        piv = piv.loc[:, piv.columns[: len(SERIES)]]
        h = max(3, 0.35 * len(piv) * max(1, piv.shape[1]) / 2)
        ax = piv.plot.barh(figsize=(8, h), color=SERIES[: piv.shape[1]], width=0.8,
                           edgecolor="white", linewidth=1)
        ax.legend(frameon=False, loc="best")
    else:
        h = max(3, 0.3 * len(tbl))
        ax = tbl.set_index("key")["value"].plot.barh(figsize=(8, h), color=SERIES[0])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel("value")
    plt.tight_layout()
    plt.show()


metric_bars(eval_tbl, "Evaluation metrics")

## Preference results (LLM judge and human)

In [ ]:
for fname, title in [("judge_preference.json", "LLM-judge preference"),
                     ("human_preference.json", "Human preference")]:
    data = load_json(fname)
    tbl = numeric_table(data)
    print(f"\n=== {title} ({fname}) ===")
    if tbl.empty:
        print("no numeric results (missing file or judge not run)")
        continue
    print(tbl.to_string(index=False))
    # win / tie / loss style keys -> one bar chart; otherwise generic bars
    wtl = tbl[tbl["key"].str.contains(r"(?:win|tie|loss|prefer)", case=False, regex=True)]
    metric_bars(wtl if not wtl.empty else tbl, title)